# Frontier LLM Research Engineering — Phase 0 scaled pilots

Run on Colab with a GPU runtime (**A100** preferred; **L4** for the FP8 pilot; T4 for the free-GPU
checks). Run the cells top to bottom. Every pilot writes to Google Drive and resumes if the session
disconnects: just run the same cell again.

Order matters when compute units run out: **P1 (noise floor) first**, then the others.

In [ ]:
# 1. Mount Drive (outputs survive disconnects) and choose where pilots write.
from google.colab import drive
drive.mount('/content/drive')
PILOT_DIR = '/content/drive/MyDrive/frontier-llm-pilots'
import os; os.makedirs(PILOT_DIR, exist_ok=True); os.environ['PILOT_DIR'] = PILOT_DIR
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# 2. Get the course code. Option A: the GitHub repo (set REPO_URL; a private repo needs a token in the URL).
#    Option B: upload a zip of the repo to Drive and set ZIP_PATH.
REPO_URL = 'https://github.com/tal-giladi/frontier-llm-research-course.git'
ZIP_PATH = ''   # e.g. '/content/drive/MyDrive/frontier-llm-research-course.zip'
import os
if ZIP_PATH:
    !rm -rf /content/course && mkdir -p /content/course && unzip -q "$ZIP_PATH" -d /content/course
    ROOT = '/content/course/' + os.listdir('/content/course')[0] if len(os.listdir('/content/course')) == 1 else '/content/course'
else:
    !rm -rf /content/course && git clone -q "$REPO_URL" /content/course
    ROOT = '/content/course'
%cd $ROOT
!git log --oneline -1 2>/dev/null || true

In [ ]:
# 3. Install the pinned stack (references/versions.md). PyTorch 2.14.1 CUDA wheel, then the lab package.
!pip -q install torch==2.14.1 --index-url https://download.pytorch.org/whl/cu128
!pip -q install numpy==2.5.3 scipy transformers==5.18.0 safetensors tokenizers==0.23.2 datasets==5.0.1 matplotlib==3.11.2 pytest==9.1.1 pyyaml==6.0.3 torchao==0.18.0
!pip -q install -e labs/common
import torch; print(torch.__version__, torch.cuda.get_device_name(0))
!python -m pytest labs/common/tests/test_model.py -x -q 2>&1 | tail -3   # quick check; the full suite runs on CPU in the build

In [ ]:
# 4. Data-v0 at pilot size (vocab 32768). ~800k documents is ~0.8B train tokens: enough for the P1 ladder.
#    Writes to Drive; skipped if already there. Expect roughly 30-60 minutes and ~2 GB.
DATA = f'{PILOT_DIR}/data-v0-32k'
import os
if not os.path.exists(f'{DATA}/meta.json'):
    !python -m frontierlab.data.prepare --docs 800000 --vocab 32768 --out "$DATA"
# Lab data and every runs/ folder live on Drive, so a disconnect or a new session loses nothing.
!mkdir -p "$PILOT_DIR/labdata" "$PILOT_DIR/runs" && rm -rf labs/common/data runs
!ln -sfn "$PILOT_DIR/labdata" labs/common/data && ln -sfn "$PILOT_DIR/runs" runs
!rm -rf labs/common/data/v0 && ln -sfn "$DATA" labs/common/data/v0
!cat "$DATA/meta.json" | head -40

## P2 — Module 2: from FLOPs to time, profiling, validating a claim

Each script compares the GPU against its datasheet and writes JSON/trace files to Drive. About
30 minutes in total on an A100. Keep `trace.json` and `mem.pickle`: they become the course-provided
traces for lesson 02.2.

In [ ]:
import torch, os
GPU = torch.cuda.get_device_name(0)
HW = 'A100-SXM-80GB' if ('A100' in GPU and '80GB' in GPU) else 'A100-SXM-40GB' if 'A100' in GPU else 'L4' if 'L4' in GPU else 'T4' if 'T4' in GPU else 'H100-SXM'
FAST = HW != 'T4'
DT, PRE, BS, SQ = ('bf16', 'baseline0', 8, 1024) if FAST else ('fp32', 'pilot-10m', 8, 512)
OUT = f'{PILOT_DIR}/p2-{HW}'; os.makedirs(OUT, exist_ok=True)
os.environ['LAB_TARGET'] = 'solution'
print(GPU, HW, DT, PRE)
!python curriculum/pilots/once.py P2 0 'python labs/module-02/lesson-01/measure.py --device cuda --dtype $DT --preset $PRE --vocab 32768 --batch $BS --seq $SQ --hw $HW 2>&1 | tee "$OUT/02-1-measure.txt"'

In [ ]:
!python curriculum/pilots/once.py P2 1 'python labs/module-02/lesson-02/profile_step.py --device cuda --dtype $DT --preset $PRE --vocab 32768 --batch $BS --seq $SQ --trace "$OUT/trace.json" --snapshot "$OUT/mem.pickle" --compile 2>&1 | tee "$OUT/02-2-profile.txt"'

In [ ]:
!python curriculum/pilots/once.py P2 2 'python labs/module-02/lesson-04/compare_ce.py --device cuda --dtype $DT --preset $PRE --vocab 32768 --batch $BS --seq $SQ --steps 40 --max-batch --out "$OUT/02-4-result.json" 2>&1 | tee "$OUT/02-4-compare.txt"'

Lesson 02.3 (multi-GPU DDP/FSDP2) is **not piloted** on Colab (single GPU). If a Kaggle 2× T4
session is available, run there:
`python labs/module-02/lesson-03/run_dp.py --device cuda --world 2 --preset pilot-10m --batch 8 --seq 512 --buckets 1 25`

In [ ]:
# Last cell: marks P2 complete when no command of it has failed; the status notebook reads this.
import glob, os
failed = glob.glob(f'{PILOT_DIR}/.failed/P2-*')
if failed:
    print('P2 NOT complete; failed commands (rerun this notebook to retry):', failed)
else:
    open(f'{PILOT_DIR}/.done/P2-COMPLETE', 'w').write('ok')
    print('P2 complete')